In [ ]:
# -- Cell 1 -- rclone + Drive. Same pattern as the previous notebooks.# Requires: Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN.import os, subprocessr = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)if r.returncode not in (0, 3):    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)from kaggle_secrets import UserSecretsClienttoken = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")os.makedirs("/root/.config/rclone", exist_ok=True)with open("/root/.config/rclone/rclone.conf", "w") as f:    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")REMOTE = "drive:Distillation"out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)print(out.stdout or out.stderr)assert out.returncode == 0, "cannot see " + REMOTE

In [ ]:
# -- Cell 2 -- deps.## PAMPA is FULL finetuning, so peft is NOT needed here (unlike the CellPPD# reproduction). But their finetune_ensemble.py imports training.adapters.common,# which pulls in rdkit, and it does the same transformers-5-only# TokenizersBackend import at module level as the classification script.subprocess.run('pip install -q -U "transformers>=5.0" "lightning>=2.4" rdkit',               shell=True, check=True)subprocess.run("pip uninstall -y -q torchao", shell=True)   # same guard as beforeimport torch, numpy as np, pandas as pd, glob, json, timeimport transformers, importlibmod = importlib.import_module("transformers.tokenization_utils_tokenizers")assert hasattr(mod, "TokenizersBackend"), "transformers too old"print("transformers", transformers.__version__, "| torch", torch.__version__)for i in range(torch.cuda.device_count()):    p = torch.cuda.get_device_properties(i)    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))assert torch.cuda.device_count() >= 2, \    "this notebook pins one model per GPU -- set Accelerator to T4 x2"

In [ ]:
# -- Cell 3 -- pull code, their repo, the student init, and the trained run.WORK = "/kaggle/working"CODE, REPO, INIT, RUNS = (WORK + "/distill", WORK + "/their_repo",                          WORK + "/init", WORK + "/runs")def pull(remote, local, extra=""):    os.makedirs(local, exist_ok=True)    subprocess.run("rclone copy %s/%s %s --transfers 16 %s -P" % (REMOTE, remote, local, extra),                   shell=True, check=True)if not os.path.exists(CODE + "/run_pampa.py"):    pull("distill", CODE)# Only data/ and training/ -- figure_generation is 401 files this never reads.for sub in ("data", "training"):    if not os.path.isdir(REPO + "/" + sub):        pull("their_repo/" + sub, REPO + "/" + sub)if not os.path.exists(INIT + "/model.safetensors"):    pull("models/peptideclm-2-mlm-small", INIT)# The distillation checkpoints. latest.pt is ~370 MB per arm.if not os.path.exists(RUNS + "/treatment/latest.pt"):    pull("results/distill", RUNS, "--include '*.pt' --include '*.json'")SCRIPT = REPO + "/training/01_regression_benchmarks_training_code/finetune_ensemble.py"DATA = REPO + "/data/PAMPA_clusters.csv"for p in (CODE + "/run_pampa.py", CODE + "/export_student.py", SCRIPT, DATA,          INIT + "/model.safetensors", RUNS + "/treatment/latest.pt"):    assert os.path.exists(p), "missing: " + p    print("ok", p.replace(WORK + "/", ""))df = pd.read_csv(DATA)print("\nPAMPA %d molecules | clusters %s"      % (len(df), df.cluster.value_counts().sort_index().to_dict()))

In [ ]:
# -- Cell 4 -- export the two models as plain HuggingFace directories.## Their script does AutoModel.from_pretrained(name, trust_remote_code=True), so# the student must look like a released checkpoint: backbone weights only, with# config.py / ChemPepMTR.py / tokenizer beside them. The MTR head is dropped --# their regression script builds its own head.## The folder names MUST contain "-small": resolve_model_scale() substring-matches# the name to choose batch size (16) and learning rate (1e-5). A differently named# folder silently gets base-scale defaults and the arms stop being comparable.EXPORT = WORK + "/exported"r = subprocess.run(["python", "export_student.py", "--runs", RUNS, "--init", INIT,                    "--out", EXPORT, "--which", "latest.pt"],                   cwd=CODE, capture_output=True, text=True)print(r.stdout[-2500:])if r.returncode != 0:    print(r.stderr[-2500:])assert r.returncode == 0, "export failed"WARM = EXPORT + "/peptideclm-2-mlm-small-warmstart"TREAT = EXPORT + "/peptideclm-2-mlm-small-treatment"# Confirm the exported treatment really is the trained model, not a copy of init.from safetensors.torch import load_filew, t = load_file(WARM + "/model.safetensors"), load_file(TREAT + "/model.safetensors")same = sum(1 for k in w if torch.equal(w[k], t[k]))print("\ntensors identical between warm-start and treatment: %d/%d (expect 0)"      % (same, len(w)))assert same == 0, "treatment export matches init -- wrong checkpoint?"

In [ ]:
# -- Cell 5 -- the run: one model per GPU, both clusters each.##   GPU 0  warm-start  clusters 1 and 6   (their released 32M, untouched)#   GPU 1  treatment   clusters 1 and 6   (distilled)## 20 jobs total, 10 per card. Cluster 1 -> test fold 0 (1,546 held out),# cluster 6 -> test fold 5 (494 held out); each has 5 inner validation folds whose# predictions get ensembled, matching their nested-CV protocol.## Splitting by MODEL rather than by cluster keeps both arms on identical hardware# and finishing together, which matters because we are comparing them to each other.## NOTE ON THE BUDGET: the first run stopped every job on `max_steps=10000`
# reached (~36 epochs); patience never fired and validation was still
# improving. In their script max_steps sets BOTH the hard cap and the LR decay
# horizon, so both are raised together here. Keep max_steps above one epoch
# (~292 steps): below that, training stops mid-epoch, val_rmse is never
# logged, and their EarlyStopping raises.
OUT = WORK + "/pampa"r = subprocess.run(["python", "-u", "run_pampa.py",                    "--repo-root", REPO, "--script", SCRIPT, "--data-csv", DATA,                    "--models", WARM, TREAT,                    "--out", OUT, "--clusters", "1", "6", "--gpus", "0", "1",                    "--split-by", "model", "--seed", "101",                    "--max-epochs", "100", "--max-steps", "36000", "--patience", "20"],                   cwd=CODE)print("exit", r.returncode)

In [ ]:
# -- Cell 6 -- results, and what they mean.res_path = OUT + "/pampa_metrics.csv"if os.path.exists(res_path):    res = pd.read_csv(res_path)    pd.set_option("display.width", 200)    print(res.to_string(index=False))    piv = res.pivot_table(index="cluster", columns="model", values="r2")    print("\nR2 by held-out cluster:"); print(piv.round(4).to_string())    if piv.shape[1] == 2:        wcol = [c for c in piv.columns if "warmstart" in c][0]        tcol = [c for c in piv.columns if "treatment" in c][0]        print("\ntreatment - warm-start, per cluster:")        for cl in piv.index:            print("   cluster %s: %+.4f" % (cl, piv.loc[cl, tcol] - piv.loc[cl, wcol]))        print("   mean:      %+.4f" % (piv[tcol] - piv[wcol]).mean())else:    print("no metrics written -- check", OUT + "/gpu*.log")print("""Reference points (paper, cluster-held-out R2):    32M MLM  0.13     32M MTR  0.38     337M  0.58     MorganFP  ~0.30Pre-registered thresholds:    < +0.03   no effect    >= +0.05  distillation transferred something    >= 0.25   strong (closes ~25% of the 0.13 -> 0.58 gap)Caveat this run cannot escape: warm-start vs treatment differs by BOTH the teachersignal AND 640M extra training tokens. A positive delta here does not separatethose. The control arm (same tokens, no teacher) is what separates them, and it isnot in this run.""")subprocess.run("rclone copy %s %s/results/pampa --drive-chunk-size 64M -P"               % (OUT, REMOTE), shell=True, check=True)print("uploaded to " + REMOTE + "/results/pampa")